# Valid failure overview

Looks across MNIST, CelebA, and SGSM. This is the honest overview table: pass rate only matters together with precondition match.

In [ ]:
from pathlib import Path
import csv, subprocess

REPO = 'https://github.com/casparbreloh/rbt4dnn-seminar.git'
DATA = Path('data/results.csv')
roots = [Path.cwd(), *Path.cwd().parents]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_base = Path('/content/drive/MyDrive/Studium/Seminar - RBT4DNN')
    roots += [drive_base, drive_base / 'experiments']
except Exception:
    pass

def find_data():
    for root in roots:
        if (root / DATA).exists():
            return root / DATA
        if (root / 'results.csv').exists():
            return root / 'results.csv'

path = find_data()
if path is None:
    repo = Path('/content/rbt4dnn-seminar')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO, str(repo)], check=True)
    path = repo / DATA

if not path.exists():
    raise FileNotFoundError(f'Could not find {DATA}')

with path.open(newline='') as f:
    rows = list(csv.DictReader(f))

print(path)


In [1]:
def f(x):
    return None if x == '' else float(x)

items = []
for r in rows:
    p = f(r['pass_rate_mean'])
    m = f(r['precondition_match_mean'])
    n = int(r['n_images']) if r['n_images'] else None
    if p is None or m is None or n is None:
        continue
    valid_failures = n * m * (1 - p)
    fp_estimate = (1 - m) * (1 - p)
    items.append((valid_failures, fp_estimate, r))

print('largest estimated valid-failure yields')
print('dataset req variant | pass | match | valid failures | fp estimate')
for valid_failures, fp_estimate, r in sorted(items, key=lambda x: x[0], reverse=True)[:12]:
    print(f"{r['dataset']} {r['requirement']} {r['variant']} | {float(r['pass_rate_mean']):.3f} | {float(r['precondition_match_mean']):.3f} | {valid_failures:.1f} | {fp_estimate:.3f}")


largest estimated valid-failure yields
dataset req variant | pass | match | valid failures | fp estimate
sgsm S2 lr | 0.134 | 0.850 | 7364.4 | 0.130
mnist M3 lr | 0.694 | 0.958 | 2930.5 | 0.013
sgsm S1 lr | 0.653 | 0.760 | 2633.4 | 0.083
celeba C6 lr | 0.796 | 0.981 | 2004.2 | 0.004
celeba C2 lr | 0.879 | 0.964 | 1170.3 | 0.004
sgsm S3 lr | 0.887 | 0.783 | 884.8 | 0.025
celeba C3 lr | 0.914 | 0.975 | 837.5 | 0.002
celeba C4 lr | 0.909 | 0.894 | 813.5 | 0.010
celeba C1 lr | 0.916 | 0.961 | 809.2 | 0.003
mnist M2 lr | 0.972 | 0.980 | 270.5 | 0.001
celeba C5 lr | 0.975 | 0.955 | 240.7 | 0.001
mnist M6 lr | 0.977 | 0.957 | 217.2 | 0.001


Rows without precondition-match data, such as M7/C7 in the local artifact, are intentionally excluded from computed valid-failure estimates.